In [0]:
%run ../00_setup/03-project-config

I initialized the control table by scanning the monthly landing folders and registering each monthly CSV as a pending batch. I used a Delta MERGE so rerunning the initialization notebook does not create duplicate batch records. This control table will drive incremental ingestion because Bronze will only process batches with pending status.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

batch_root_path = f"{landing_volume_path}/raw/energy"
batch_control_table = f"{catalog_name}.{audit_schema}.batch_control"

batch_records = []

year_folders = dbutils.fs.ls(batch_root_path)

for year_folder in year_folders:
    if not year_folder.name.startswith("year="):
        continue

    batch_year = int(year_folder.name.replace("year=", "").replace("/", ""))

    month_folders = dbutils.fs.ls(year_folder.path)

    for month_folder in month_folders:
        if not month_folder.name.startswith("month="):
            continue

        batch_month = int(month_folder.name.replace("month=", "").replace("/", ""))
        month_text = str(batch_month).zfill(2)

        source_file_name = f"germany_energy_{batch_year}_{month_text}.csv"
        source_path = f"{month_folder.path}{source_file_name}"
        batch_id = f"{batch_year}_{month_text}"

        batch_records.append((
            batch_id,
            batch_year,
            batch_month,
            source_path,
            source_file_name,
            "pending",
            None,
            None,
            None,
            None,
            None,
            0,
            None,
            None
        ))

schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("batch_year", IntegerType(), False),
    StructField("batch_month", IntegerType(), False),
    StructField("source_path", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("status", StringType(), False),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("retry_count", IntegerType(), False),
    StructField("created_timestamp", TimestampType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

batch_df = spark.createDataFrame(batch_records, schema)

batch_df.createOrReplaceTempView("new_batch_records")

spark.sql(f"""
MERGE INTO {batch_control_table} AS target
USING (
    SELECT
        batch_id,
        batch_year,
        batch_month,
        source_path,
        source_file_name,
        status,
        start_timestamp,
        end_timestamp,
        rows_read,
        rows_written,
        error_message,
        retry_count,
        current_timestamp() AS created_timestamp,
        current_timestamp() AS updated_timestamp
    FROM new_batch_records
) AS source
ON target.batch_id = source.batch_id
WHEN NOT MATCHED THEN
    INSERT *
""")

print("batch_control initialized successfully")

In [0]:
display(spark.sql(f"""
SELECT
    batch_id,
    batch_year,
    batch_month,
    source_file_name,
    status,
    retry_count,
    created_timestamp
FROM {batch_control_table}
ORDER BY batch_year, batch_month
"""))

display(spark.sql(f"""
SELECT
    status,
    COUNT(*) AS batch_count
FROM {batch_control_table}
GROUP BY status
"""))